# Notebook 08 — Interactive Design Selection

**Learning objectives:**
- Use interactive sliders to filter designs by metric thresholds
- Explore the trade-offs between different quality criteria
- Finalize your top 5 selection for the Design Recommendation Report
- Export a formatted report table

**Time:** ~45 minutes

**Companion wiki pages:** [7.2 Ranking](https://rucha1796.github.io/internship-bindcraft-2026/m7_02_ranking/) | [7.4 Case Study](https://rucha1796.github.io/internship-bindcraft-2026/m7_04_case_study/)

---
> No GPU required. Run after completing Notebooks 05–07.

In [ ]:
!pip install ipywidgets -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

print("✓ Ready")

## Cell 1 — Load data

In [ ]:
DATA_DIR = "/content/drive/MyDrive/bindcraft/PDL1_8aok"
csv_path = f"{DATA_DIR}/final_design_stats.csv"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"✓ Loaded {len(df)} designs")
else:
    print("Generating demo dataset...")
    np.random.seed(42)
    n = 15
    df = pd.DataFrame({
        "binder_name": [f"PDL1_t{i}_mpnn1" for i in range(1, n+1)],
        "binder_sequence": ["MKVIFGLMVGGVIAK" + "AEK" * i for i in range(1, n+1)],
        "Average_i_pTM": np.random.uniform(0.55, 0.76, n),
        "Average_pLDDT": np.random.uniform(0.85, 0.95, n),
        "Average_Binder_pLDDT": np.random.uniform(0.80, 0.93, n),
        "Average_Binder_RMSD": np.random.uniform(0.4, 2.0, n),
        "Average_ShapeComplementarity": np.random.uniform(0.55, 0.73, n),
        "Average_dG": np.random.uniform(-26, -10, n),
        "Average_i_pAE": np.random.uniform(4.5, 9.8, n),
        "Average_n_InterfaceHbonds": np.random.randint(3, 14, n).astype(float),
        "Average_Relaxed_Clashes": np.random.uniform(0, 4.5, n),
    })
    print(f"✓ Demo dataset: {len(df)} designs")

KEY_METRICS = [c for c in [
    "Average_i_pTM", "Average_pLDDT", "Average_Binder_pLDDT",
    "Average_Binder_RMSD", "Average_ShapeComplementarity",
    "Average_dG", "Average_i_pAE", "Average_n_InterfaceHbonds", "Average_Relaxed_Clashes"
] if c in df.columns]

## Cell 2 — Interactive filter sliders

Move the sliders to explore how different threshold choices affect how many designs survive. The table updates in real time.

In [ ]:
# Slider widgets for key filters
slider_iptm = widgets.FloatSlider(value=0.55, min=0.40, max=0.75, step=0.01,
    description="Min i_pTM:", style={"description_width": "140px"}, layout=widgets.Layout(width="500px"))
slider_rmsd = widgets.FloatSlider(value=2.0, min=0.5, max=3.5, step=0.1,
    description="Max Binder RMSD:", style={"description_width": "140px"}, layout=widgets.Layout(width="500px"))
slider_sc = widgets.FloatSlider(value=0.55, min=0.35, max=0.75, step=0.01,
    description="Min ShapeComp:", style={"description_width": "140px"}, layout=widgets.Layout(width="500px"))
slider_dg = widgets.FloatSlider(value=-10.0, min=-30.0, max=-5.0, step=0.5,
    description="Max dG:", style={"description_width": "140px"}, layout=widgets.Layout(width="500px"))

output = widgets.Output()

def apply_filters(iptm_min, rmsd_max, sc_min, dg_max):
    """Apply slider thresholds and display filtered designs."""
    mask = pd.Series([True] * len(df))
    
    if "Average_i_pTM" in df.columns:
        mask &= df["Average_i_pTM"] >= iptm_min
    if "Average_Binder_RMSD" in df.columns:
        mask &= df["Average_Binder_RMSD"] <= rmsd_max
    if "Average_ShapeComplementarity" in df.columns:
        mask &= df["Average_ShapeComplementarity"] >= sc_min
    if "Average_dG" in df.columns:
        mask &= df["Average_dG"] <= dg_max
    
    filtered = df[mask].sort_values("Average_i_pTM", ascending=False)
    
    with output:
        clear_output(wait=True)
        print(f"Designs passing current thresholds: {len(filtered)} / {len(df)}")
        print(f"  i_pTM ≥ {iptm_min:.2f} | Binder RMSD ≤ {rmsd_max:.1f} | ShapeComp ≥ {sc_min:.2f} | dG ≤ {dg_max:.1f}")
        print()
        if len(filtered) > 0:
            show_cols = ["binder_name"] + [c for c in KEY_METRICS if c in df.columns]
            pd.set_option("display.float_format", "{:.3f}".format)
            print(filtered[show_cols].to_string(max_rows=10))
            if len(filtered) > 10:
                print(f"  ... ({len(filtered) - 10} more)")
        else:
            print("  No designs pass these thresholds. Try relaxing the sliders.")

# Wire up sliders
def on_change(change):
    apply_filters(
        slider_iptm.value, slider_rmsd.value,
        slider_sc.value, slider_dg.value
    )

for slider in [slider_iptm, slider_rmsd, slider_sc, slider_dg]:
    slider.observe(on_change, names="value")

# Display UI
display(widgets.VBox([
    widgets.HTML("<b>Adjust thresholds — table updates automatically:</b>"),
    slider_iptm, slider_rmsd, slider_sc, slider_dg,
    output
]))

# Initial render
apply_filters(slider_iptm.value, slider_rmsd.value, slider_sc.value, slider_dg.value)

## Cell 3 — Interactive scatter plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

if "Average_Binder_RMSD" in df.columns:
    sc = ax.scatter(
        df["Average_i_pTM"],
        df["Average_ShapeComplementarity"],
        c=df["Average_Binder_RMSD"],
        cmap="RdYlGn_r", vmin=0.4, vmax=2.0,
        s=150, edgecolors="black", linewidth=0.7, zorder=3
    )
    plt.colorbar(sc, ax=ax, label="Binder RMSD (Å)")
else:
    ax.scatter(df["Average_i_pTM"], df["Average_ShapeComplementarity"],
               s=150, color="#3c5b6f", edgecolors="black")

# Annotate each point with design index
df_sorted = df.sort_values("Average_i_pTM", ascending=False).reset_index(drop=True)
for idx, (i_ptm, sc_val) in enumerate(
    zip(df_sorted["Average_i_pTM"], df_sorted["Average_ShapeComplementarity"]), start=1
):
    ax.annotate(str(idx), xy=(i_ptm, sc_val), xytext=(4, 4),
                textcoords="offset points", fontsize=9, fontweight="bold", color="navy")

# Draw threshold lines
ax.axvline(0.55, color="red", linestyle="--", alpha=0.6, label="i_pTM threshold (0.55)")
ax.axhline(0.55, color="orange", linestyle="--", alpha=0.6, label="ShapeComp threshold (0.55)")
ax.axvline(0.65, color="darkred", linestyle=":", alpha=0.5, label="i_pTM strong (0.65)")
ax.axhline(0.65, color="darkorange", linestyle=":", alpha=0.5, label="ShapeComp strong (0.65)")

ax.set_xlabel("Average i_pTM", fontsize=12)
ax.set_ylabel("Average Shape Complementarity", fontsize=12)
ax.set_title("Design Space: i_pTM vs. ShapeComplementarity\n(Numbers = rank by i_pTM | Color = Binder RMSD)", fontsize=11)
ax.legend(fontsize=9, loc="lower right")

# Shade the top-right zone (best designs)
ax.fill_between([0.65, ax.get_xlim()[1]], 0.65, ax.get_ylim()[1],
                alpha=0.05, color="green", label="excellent zone")

plt.tight_layout()
plt.savefig("design_space.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: design_space.png")
print("Top-right + green color = ideal designs (high i_pTM + high ShapeComp + low RMSD)")

## Cell 4 — Finalize your top 5 selection

Enter your chosen 5 design names below. The cell will pull all metrics and format them for your Design Recommendation Report.

In [ ]:
# ============================================================
# ENTER YOUR TOP 5 DESIGN NAMES HERE
# (copy from the table in Cell 2 or Cell 3 above)
TOP5_NAMES = [
    # "PDL1_trajectory1_mpnn2",    # ← example; replace with your actual names
    # "PDL1_trajectory4_mpnn1",
    # "PDL1_trajectory7_mpnn3",
    # "PDL1_trajectory9_mpnn1",
    # "PDL1_trajectory12_mpnn2",
]
# ============================================================

if not TOP5_NAMES:
    # Auto-select top 5 if user hasn't filled in names yet
    TOP5_NAMES = df.sort_values("Average_i_pTM", ascending=False).head(5)["binder_name"].tolist()
    print("⚠ No names entered — using top 5 by i_pTM automatically.")
    print("  Edit TOP5_NAMES in this cell with your chosen designs.")
    print()

top5_df = df[df["binder_name"].isin(TOP5_NAMES)].copy()

if len(top5_df) == 0:
    print("⚠ None of the entered names found in dataset. Check spelling.")
else:
    top5_df = top5_df.sort_values("Average_i_pTM", ascending=False).reset_index(drop=True)
    top5_df.index += 1
    
    print("=" * 80)
    print("YOUR TOP 5 SELECTIONS")
    print("=" * 80)
    
    display_cols = [c for c in [
        "binder_name", "Average_i_pTM", "Average_ShapeComplementarity",
        "Average_Binder_RMSD", "Average_dG", "Average_n_InterfaceHbonds"
    ] if c in top5_df.columns]
    
    pd.set_option("display.float_format", "{:.3f}".format)
    print(top5_df[display_cols].to_string())

## Cell 5 — Generate Design Recommendation Report

In [ ]:
from datetime import date

print("=" * 70)
print("PD-L1 BINDER DESIGN RECOMMENDATION REPORT")
print(f"Date: {date.today()}")
print(f"Intern: [YOUR NAME HERE]")
print(f"Dataset: PDL1_8aok")
print(f"Total accepted designs reviewed: {len(df)}")
print("=" * 70)
print()

print("TOP 5 DESIGNS RECOMMENDED FOR SYNTHESIS")
print("-" * 70)

for rank, (_, row) in enumerate(top5_df.iterrows(), start=1):
    print(f"\nRank {rank}: {row.get('binder_name', 'N/A')}")
    print(f"  Interface confidence (i_pTM):   {row.get('Average_i_pTM', 0):.3f}")
    print(f"  Shape complementarity:          {row.get('Average_ShapeComplementarity', 0):.3f}")
    print(f"  Binder RMSD (self-consistency): {row.get('Average_Binder_RMSD', 0):.2f} Å")
    print(f"  Predicted binding energy (dG):  {row.get('Average_dG', 0):.1f} kcal/mol")
    print(f"  Interface H-bonds:              {row.get('Average_n_InterfaceHbonds', 0):.0f}")
    if "binder_sequence" in row:
        seq = row["binder_sequence"]
        print(f"  Sequence ({len(seq)} aa):           {seq}")
    print(f"  Justification: [Edit this cell — why is this design recommended?]")

print()
print("-" * 70)
print("NEXT STEPS")
print("-" * 70)
print("1. Gene synthesis: order the 5 sequences above from Twist Bioscience")
print("2. Expression: E. coli BL21(DE3), His-tag, IPTG induction")
print("3. Testing: ELISA (week 1) → SPR if ELISA positive (week 2-3)")
print("4. Success criterion: Kd < 100 nM in SPR, preferably < 10 nM")
print()
print("SUGGESTIONS FOR NEXT DESIGN ROUND")
print("-" * 70)
print("[Edit this section based on your failure analysis from Notebook 07]")

## Cell 6 — Export report to text file

In [ ]:
# Export the top 5 table as CSV for easy sharing
report_csv_path = "design_recommendation_report.csv"

report_cols = [c for c in [
    "binder_name", "binder_sequence",
    "Average_i_pTM", "Average_ShapeComplementarity",
    "Average_Binder_RMSD", "Average_pLDDT",
    "Average_Binder_pLDDT", "Average_dG",
    "Average_n_InterfaceHbonds", "Average_Relaxed_Clashes",
    "Average_i_pAE"
] if c in top5_df.columns]

top5_df[report_cols].to_csv(report_csv_path, index=True)
print(f"✓ Saved: {report_csv_path}")

# Download to your computer
try:
    from google.colab import files
    files.download(report_csv_path)
    print("  Downloading to your computer...")
except Exception:
    print(f"  (To download: right-click '{report_csv_path}' in the Files panel → Download)")

---
## Congratulations!

You have completed the PD-L1 binder design workflow:

1. ✅ Learned protein biology and the PD-L1/PD-1 checkpoint
2. ✅ Ran AlphaFold2 on PD-L1 (NB02)
3. ✅ Ran BindCraft to design binders (NB04)
4. ✅ Analyzed the output with pandas and plots (NB05)
5. ✅ Visualized your designs in 3D (NB06)
6. ✅ Analyzed failure patterns (NB07)
7. ✅ Selected your top 5 and wrote the Design Recommendation Report (NB08)

**If you are on the 4-week track:** Your report is your final deliverable. Present it to your mentor.

**If you are on the 6-week track:** Continue to Notebook 09 — Independent Project Template.

**Next:** Notebook 09 — Your independent target